In [47]:
from pathlib import Path

import polars as pl
import polars_distance as pld
import xgboost as xgb
from geopy.distance import geodesic
from sentence_transformers import SentenceTransformer
from xgboost import XGBClassifier

from ml_deduplication.modeling.splink_model import strip_ville_from_name
from ml_deduplication.training.utils import split_train_dev

In [2]:
RANDOM_SEED = 42

In [3]:
DATASETS_PATH = Path("../../datasets/")

# Chargement des données

In [4]:
df_features = pl.read_parquet(
    DATASETS_PATH / "features_dataset_20260825_4k_entities.parquet"
)

# Preprocessing

In [5]:
def strip_ville_from_name(data: dict) -> str:
    if (data["ville"] is None) or (data["ville"].strip() == ""):
        return data["nom"]
    nom_clean = data["nom"].replace(data["ville"].strip(), "")

    if data["ville_clean"] is not None:
        nom_clean = data["nom"].replace(data["ville_clean"].strip(), "")

    return nom_clean


def preprocess_features(
    df_features: pl.DataFrame, embedding_model: SentenceTransformer
) -> pl.DataFrame:
    df_features_preprocessed = df_features.clone()

    df_features_preprocessed = df_features_preprocessed.with_columns(
        pl.selectors.by_dtype(pl.String)
        .exclude(["cluster_id", "cluster_id_split", "label"])
        .str.strip_chars()
        .replace("__empty__", None)
        .replace("", None)
    )
    str_column_to_process = [
        "nom",
        "nom_commercial",
        "ville",
    ]

    df_features_preprocessed = (
        df_features_preprocessed.with_columns(
            pl.col(e)
            .str.strip_chars()
            .str.to_lowercase()
            .str.normalize("NFKD")
            .map_elements(lambda x: x.encode("ASCII", "ignore").decode("utf-8"))
            for e in str_column_to_process
        )
        .with_columns(
            pl.col("ville")
            .str.replace_all("st", "saint", literal=True)
            .str.replace_all("-", " ", literal=True)
            .alias("ville_clean")
        )
        .with_columns(
            pl.selectors.starts_with("latitude").clip(-90.0, 90.0),
            pl.selectors.starts_with("longitude").clip(-180.0, 180.0),
            pl.struct(
                pl.concat_str(
                    "nom", "nom_commercial", separator=" ", ignore_nulls=True
                ).alias("nom"),
                "ville",
                "ville_clean",
            )
            .map_elements(strip_ville_from_name, return_dtype=pl.String)
            .alias(
                "nom_clean"
            ),  # Concatenate nom and nom_commercial and eliminate ville from nom
            pl.concat_str(
                pl.col("adresse").fill_null(""),
                pl.col("adresse_complement").fill_null(""),
                separator=" ",
            ).alias(
                "adresse_clean"
            ),  # Concatenate adresse and adresse_complement
            pl.coalesce(["nom", "nom_commercial"]).alias("nom"),
            pl.coalesce(["nom_commercial", "nom"]).alias("nom_commercial"),
        )
    )

    addresses_texts = df_features_preprocessed.get_column("adresse_clean").to_list()

    addresses_tensors = embedding_model.encode(addresses_texts)

    df_vectors = pl.DataFrame({"adresse_clean_vector": addresses_tensors})
    df_features_preprocessed = pl.concat(
        [df_features_preprocessed, df_vectors], how="horizontal"
    ).with_columns(
        pl.when(pl.col("adresse").is_null() & pl.col("adresse_complement").is_null())
        .then(None)
        .otherwise(pl.col("adresse_clean_vector").cast(pl.Array(pl.Float64, 1024)))
        .alias("adresse_clean_vector")
    )

    return df_features_preprocessed

In [6]:
embedding_model = SentenceTransformer("Lajavaness/sentence-camembert-large")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [7]:
df_features_preprocessed = preprocess_features(df_features, embedding_model)

/var/folders/g4/t3006b6x41b4f5tmkgqprc0c0000gn/T/ipykernel_21376/3020934401.py:76: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  df_features_preprocessed = pl.concat(


# Blocking

In [30]:
def block_df(df_features=pl.DataFrame) -> pl.DataFrame:

    df_features = df_features.lazy()

    business_rules_filter_expr = [
        (
            (pl.col("acteur_type_id_l") == pl.col("acteur_type_id_r"))
            | ((pl.col("acteur_type_id_l") == 4) & (pl.col("acteur_type_id_r") == 3))
            | ((pl.col("acteur_type_id_l") == 3) & (pl.col("acteur_type_id_r") == 4))
        ),
        (
            pl.coalesce(pl.col("source_id_l"), pl.lit(-1))
        != pl.coalesce(pl.col("source_id_r"), pl.lit(-2))
        ),
    ]

    df_pairs = (
        df_features.rename(lambda x: x + "_l")
        .join(df_features.rename(lambda x: x + "_r"), how="cross")
        .with_columns(
            pl.struct("latitude_l", "longitude_l", "latitude_r", "longitude_r")
            .map_elements(
                lambda x: geodesic(
                    (x["latitude_l"], x["longitude_l"]),
                    (x["latitude_r"], x["longitude_r"]),
                ).km,
                return_dtype=pl.Float64,
            )
            .alias("geo_distance")
        )
        .filter(pl.col("identifiant_unique_l") < pl.col("identifiant_unique_r"))
        .filter(*business_rules_filter_expr)
        .filter(
            (
                pl.col("code_postal_l").str.slice(0, 2)
                == pl.col("code_postal_r").str.slice(0, 2)
            )
            | (pl.col("siren_l") == pl.col("siren_r"))
            | (pl.col("geo_distance") < 30)
        )
    )

    return df_pairs.collect(engine="streaming")

In [31]:
df_features_train = df_features_preprocessed.filter(pl.col("split") == "train")

In [42]:
df_features_train_sub, df_features_dev = split_train_dev(df_features_train)

In [43]:
df_pairs_train_sub = block_df(df_features_train_sub)
df_pairs_dev = block_df(df_features_dev)

In [34]:
def generate_features(df_pairs: pl.DataFrame) -> pl.DataFrame:
    df_pairs_features = df_pairs.with_columns(
        pld.col("nom_clean_l")
        .dist_str.jaro_winkler("nom_clean_r")
        .alias("nom_clean_dist"),
        pld.col("adresse_clean_vector_l")
        .dist_arr.cosine("adresse_clean_vector_r")
        .alias("adresse_clean_distance"),
        pld.col("ville_clean_l")
        .dist_str.jaro_winkler("ville_clean_r")
        .alias("ville_clean_dist"),
        (pl.col("siren_l") == pl.col("siren_r")).alias("siren_match"),
        (pl.col("siret_l") == pl.col("siret_r")).alias("siret_match"),
        (pl.col("telephone_l") == pl.col("telephone_r")).alias("telephone_match"),
        (pl.col("code_commune_insee_l") == pl.col("code_commune_insee_r")).alias(
            "code_commune_insee_match"
        ),
        (pl.col("code_postal_l") == pl.col("code_postal_r")).alias("code_postal_match"),
        (
            pl.col("code_postal_l").str.slice(0, 2)
            == pl.col("code_postal_r").str.slice(0, 2)
        ).alias("departement_match"),
        pl.coalesce((pl.col("cluster_id_l") == pl.col("cluster_id_r")), False).alias(
            "label"
        ),
    ).select(
        "identifiant_unique_l",
        "identifiant_unique_r",
        "nom_clean_dist",
        "adresse_clean_distance",
        "ville_clean_dist",
        "siren_match",
        "siret_match",
        "telephone_match",
        "code_commune_insee_match",
        "code_postal_match",
        "departement_match",
        "label",
        pl.when("label").then("cluster_id_l").otherwise(None),
    )

    return df_pairs_features

In [44]:
df_pairs_features_train_sub = generate_features(df_pairs_train_sub)
df_pairs_features_dev = generate_features(df_pairs_dev)

In [45]:
df_pairs_features_train_sub.group_by("label").len()

label,len
bool,u32
true,2579
false,31062


# Modeling

In [48]:
early_stop = xgb.callback.EarlyStopping(
    rounds=2, metric_name="aucpr", data_name="validation_0", save_best=True
)

model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    objective="binary:logistic",
    subsample=0.8,
    colsample_bytree=0.9,
    gamma=1,
    reg_alpha=0.1,
    reg_lambda=5,
    scale_pos_weight=14,
    callbacks=[early_stop],
)

In [52]:
X_train_sub = df_pairs_features_train_sub.select(
    "nom_clean_dist",
    "adresse_clean_distance",
    "ville_clean_dist",
    "siren_match",
    "siret_match",
    "telephone_match",
    "code_commune_insee_match",
    "code_postal_match",
    "departement_match",
).to_numpy()
y_train_sub = df_pairs_features_train_sub.select(
    pl.col("label").cast(pl.Float16)
).to_numpy()

X_dev = df_pairs_features_dev.select(
    "nom_clean_dist",
    "adresse_clean_distance",
    "ville_clean_dist",
    "siren_match",
    "siret_match",
    "telephone_match",
    "code_commune_insee_match",
    "code_postal_match",
    "departement_match",
).to_numpy()
y_dev = df_pairs_features_dev.select(pl.col("label").cast(pl.Float16)).to_numpy()

In [ ]:
model.fit(X_train_sub, y_train_sub, eval_set=[(X_dev, y_dev)])